# 산업 화학 공정 모니터링

프로젝트 번호: 15

# 목차


1. 프로젝트 개요
	1. 프로젝트 연구 배경
    1. SensorSCAN
    1. 기대 효과
1. 실습 환경 설정
	1. 엘리스 클라우드 가입
	1. 인스턴스 생성
	1. 인스턴스 삭제
    1. 인스턴스 접속(SSH)
    1. 버전 확인
	1. 코드 업로드
    1. 패키지 설치
1. 데이터셋 준비
    1. 데이터셋 소개
    1. 데이터셋 다운로드
    1. 데이터셋 다운로드 확인
    1. 데이터셋 시각화
1. 모델 구축
    1. 파라미터 설정
    1. 필요한 모듈 로딩
    1. 데이터셋 분할 및 로드
	1. 데이터셋 분할
	1. 데이터셋 분할 결과 확인
	1. 모델 생성 함수
	1. 모델 학습 함수
	1. 모델 평가 함수
1. 모델 학습
	1. 주요 변수 설명
	1. 모델 학습 관련 코드 블록
1. 모델 실험 결과
    1. 결과 비교 및 분석
1. 결론
	1. 주요 업적
	1. 현장 실무에 유용할 주요 핵심 정리
	1. 기존 연구와의 차별점
	1. 기술적 한계 및 향후 연구 과제

# 프로젝트 개요

예상 모델 학습 소요시간: `30분`

컴퓨팅 성능 요구: `낮음`

논문: M. Golyadkin, V. Pozdnyakov, L. Zhukov, and I. Makarov, "SensorSCAN: Self-supervised learning and deep clustering for fault diagnosis in chemical processes," Artif. Intell., vol. 324, Art. no. 104012, Nov. 2023. doi: 10.1016/j.artint.2023.104012.

소스코드 원본 주소: https://github.com/airi-institute/sensorscan

## 프로젝트 연구 배경

- 기존 기술의 문제점
    - 전통적 비지도 기법의 표현력 한계
        - PCA, FDA 같은 통계적 차원 축소 후 k-means·DBSCAN 군집화 방식은 선형관계 가정 및 저차원 표현의 제약으로, 공정의 복잡한 비선형·상호의존적 센서 패턴을 충분히 포착하지 못함

    - 기존 딥러닝 비지도 클러스터링의 초기화 취약성
        - Autoencoder, GAN 기반 또는 엔드투엔드 비지도 이미지 클러스터링(e.g. Deep Embedded Clustering)이 랜덤 초기화된 특징 추출기(F) 위에서 학습되면, 저수준 피처에만 의존해 임베딩이 조밀하게 뭉치고 고수준 의미 구조를 반영하지 못해 성능이 저하됨

    - 지도학습 기반 FDD의 레이블 의존성
        - 대부분의 데이터 기반 이상 탐지·진단(FDD)은 센서 데이터에 “정상 vs 이상” 레이블을 매핑해야 하는 지도학습 설정을 전제로 함
        - 그러나 대규모 센서 로그를 전문가가 일일이 주석 처리하는 것은 비용·시간 부담이 크고, 결함 발생 시점을 정확히 파악하기 어려워 실제 산업 현장 적용이 제한적


- 사회적·산업적 문제
    - 방대한 센서 데이터와 결함 희소성
        - 화학·제조 공정 설비는 수백~수천 개의 센서를 통해 막대한 양의 시계열 데이터를 생성하나, 결함(이상)은 전체 시간 대비 극히 드물어 정상 패턴에 가려짐

    - 공정 중단·수율 저하·장비 손상 위험
        - 결함 조기 탐지 실패 시 공정 중단, 제품 수율 감소, 설비 마모·고장으로 이어져 안전사고 및 경제적 손실을 초래

    - 레이블이 없는 실제 운영 데이터
        - 보안·프라이버시 이유로 대부분 공정 데이터는 공개되지 않고, 실제 운영 데이터는 레이블이 부재하거나 불균형해 전통 지도학습 적용이 어려움

## SensorSCAN 기법

`자기지도 학습 기반 사전학습`
- 모델은 레이블 없는 시계열 데이터로부터 자체 구조를 학습하기 위해, Transformer 인코더를 특징 추출기로 사용
- 인코더는 전역 자기-어텐션을 통해 시퀀스 전반의 복잡한 의존성을 포착하며, 이후 가중합 풀링과 프로젝션 헤드를 거쳐 고차원 임베딩을 생성
- 학습 단계에서는 마스킹된 입력 복원과 대조 학습(Contrastive Learning)을 함께 적용
- 하나는 누락된 센서 값을 정확히 예측하도록 내부 패턴 학습을 강화하과, 다른 하나는 증강된 뷰 간 유사도를 극대화해 샘플 간 구분력을 높임
- 서로 보완적으로 작용해 최종 임베딩의 견고성과 표현력을 크게 향상

`변형된 SCAN 기반 딥 클러스터링`
- 사전학습된 임베딩 위에 클러스터링 헤드를 얹고, SCAN 손실을 활용해 클러스터 예측을 학습
- SCAN 손실은 “이웃” 샘플이 같은 클러스터를 예측하도록 유도하면서, 클러스터 분포의 엔트로피 항을 통해 과도한 편향을 방지
- 특히, 시계열 슬라이딩 윈도우에서 발생하는 높은 상관성을 완화하기 위해 서브샘플링 기반 이웃 탐색을 도입
- 청크 단위로만 K-NN을 수행함으로써 불필요한 중복을 줄이고 안정적인 클러스터링을 구현

`클러스터-레이블 매핑 및 소수 라벨 미세조정`
- 학습된 클러스터에 공정 상태(정상·결함 유형)를 자동 매핑하는 Label Matching 기법을 적용
- 각 클러스터 내에서 정상 샘플이 차지하는 비율에 가중치를 두어, 전문가가 최소한의 검토만으로도 정확한 매핑이 가능하도록 설계
- 여기에, 실제 운영 데이터의 극소량 라벨만으로 전체 모델을 미세조정(Fine-tuning) 함으로써, 비지도 학습 성능을 거의 그대로 유지하면서 지도학습 수준의 결함 탐지·진단 정확도를 달성

`SensorSCAN schematic view`
<figure>
    <img src="image/sensorscan_schematicview.png" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

`SensorSCAN Method`
<figure>
    <img src="image/sensorscan_method.png" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

## 기대 효과

- 레이블 없이도 높은 결함 탐지·진단 정확도
    - 완전 비지도 상태에서도 전통 기법 대비 크게 향상된 결함 탐지 성능을 보임
    - 특히 데이터 라벨 없이 사전학습과 딥 클러스터링만으로도 결함 발생 여부를 식별하는 True Positive Rate(TPR)이 0.87에 달하며, 잘못 경보를 줄이는 False Positive Rate(FPR)은 거의 0에 근접
    - 이러한 성능은 PCA·ConvAE·ST-CatGAN 등 기존 비지도 방법의 TPR(약 0.50–0.60) 대비 최소 20–30%p 개선된 수치
    - 라벨 확보가 불가능한 환경에서도 현저히 신뢰할 수 있는 모니터링을 가능하게 함

- 조기 경보를 위한 검출 지연 시간 대폭 단축
    - 평균 검출 지연 시간(ADD)이 약 28.5 스텝
    - PCA(111.5 스텝), ST-CatGAN(135.0 스텝), ConvAE(52.3 스텝) 대비 절반 이하로 줄어든 값
    - 결함 발생 직후에도 20% 이내의 짧은 시간에 경보를 발송할 수 있어, 공정 중단이나 대규모 손실로 이어지기 전에 즉각적인 대응이 가능
    - 실제 공정 사이클(예: TEP 공정의 3분 주기)과 비교했을 때, ADD 28.5 스텝은 수 초 내외 수준에 해당하므로 실시간 모니터링 요건을 충분히 충족

- 최소한의 라벨로 지도학습급 성능 달성
    - SensorSCANsingle은 “한 번의 라벨링 런”이라 불리는 극소량(수십 개 런) 라벨 데이터로만 미세조정함
    - 라벨을 대량 사용하는 GRU 기반 풀 데이터 모델(GRUfull)과 거의 동등한 탐지 성능을 보여줌
    - 이 과정에서 전문가가 새 결함 유형에 대해 단 한 번만 라벨링하면 되므로, 라벨 확보에 드는 시간과 비용을 최대 90% 이상 절감할 수 있음
    - 결과적으로 완전 비지도 학습의 장점을 유지하면서도 지도학습 모델에 육박하는 정확도를 실현

- 미지 결함에 대한 견고한 일반화 능력
    - 사전학습된 Transformer 기반 특성 추출기는 학습 데이터에 포함되지 않은 신규 결함도 잠재 공간에서 효과적으로 분리
    - 클러스터 수를 늘리면, 종래에 혼합(clustered)되어 구분이 어려웠던 결함 유형들을 점진적으로 순수 클러스터로 분리
    - 그 결과 알려지지 않은 이상 상황에서도 높은 탐지율을 유지하거나 오히려 개선되는 경향을 보임
    - 이로써, 공정 변화나 장비 확장 시에도 추가 학습 없이 곧바로 적용할 수 있는 유연성을 제공

- 운영 부담 최소화 및 확장 용이성
    - 자동화된 Label Matching 기법을 통해, 클러스터별 정상·결함 상태 매핑은 최소한의 전문가 개입으로 완료
    - 클러스터 수를 두 배로 늘려도 추가로 필요한 수작업 라벨링량은 크게 증가하지 않음
    - 연속적 모니터링 시스템 운영 시 라벨 관리 부담이 본질적으로 경감
    - 신규 센서나 결함 유형이 추가될 때에도 전체 파이프라인을 재실행하고 소량 라벨만 보강하면 되므로 시스템 확장이 매우 용이

- 실시간 적용을 위한 연산 및 자원 효율성
    - SensorSCAN의 추론은 한 샘플당 평균 0.01초 내외
    - 초당 수십에서 수백 개의 시계열 윈도우를 처리할 수 있음
    - 모델 크기와 클러스터 헤드 구조, 증강·클러스터링 파라미터 조합만으로 연산 복잡도와 메모리 사용량을 유연하게 조절할 수 있음
    - 엣지 디바이스부터 서버급 환경에 이르기까지 다양한 하드웨어에서 실시간 모니터링을 구현할 수 있음

# 실습 환경 설정

## 엘리스 클라우드 가입

1. 엘리스 클라우드 가입 시 사용할 이메일 주소로 초대장이 보내짐
1. 이메일을 열어 초대장 메일을 확인

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/mail1.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

3. "`학습지 가기`"를 클릭하여 로그인 화면으로 이동

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/mail2.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

4. 메일에 나와있는 이메일 아이디와 임시포스트를 입력하여 로그인

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/login.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

5. 사용할 비밀번호를 입력하고 필수 동의사항에 동의 후 회원가입 클릭

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/login2.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

6. 회원가입 완료

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/login3.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

## 인스턴스 생성

1. 왼쪽 상단의 LXP를 클릭한 뒤 클라우드를 클릭

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/cloud.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

2. 내 인스턴스 클릭

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/cloud2.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

3. 인스턴스 생성을 클릭하여 생성 화면으로 이동

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/cloud3.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

4. 인스턴스 유형으로 G-NAHP-80(A100 80GB PCle)을 선택

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/instance.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

5. 인스턴스 이름 작성 후 실행 환경은 Jupyter 선택

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/instance2.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

6. 스토리지 옵션으로 `{프로젝트 용량 고려}`GB 선택\
7. 이후 인스턴스 생성을 클릭해 생성을 완료

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/instance3.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

8. 이제 생성된 인스턴스에서 실습을 진행함
1. 이후 내 인스턴스 페이지로 돌아가기 위해 왼쪽 상단의 내 인스턴스 클릭

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/instance4.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

## 인스턴스 삭제

- 인스턴스 화면에 없어도 인스턴스는 계속 실행되고 있으므로 삭제 필요
- 사용이 끝난 경우 반드시 ‘종료’ 버튼을 눌러 실행을 중지해야 함\
(단, 종료는 인스턴스에 여전히 필요한 데이터나 설정이 남아 있는 경우에만 사용)
- 인스턴스를 더 이상 사용할 일이 없다면, ‘삭제’를 통해 완전히 제거해야함
- 삭제는 반드시 인스턴스가 종료된 상태에서만 가능하며, 한 번 삭제하면 복구할 수 없음

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/instance5.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

## 인스턴스 접속(SSH)

### PEM 다운로드

> 주의: 한번 다운로드 받은 PEM 파일은 잘 보관해야 함\
> 재발급 받은 경우 키 값이 달라지므로 재발급 받은 파일로는 기존의 인스턴스에 접속 불가함\
> SSH 접속은 실습 환경에 필요한 모듈을 다운받는 등에 사용함

1. SSH 접속을 위해서는 비밀키 파일(PEM 파일) 필요\
비밀키 발급하기 클릭

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/key.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

2. SSH 접속을 위해서는 비밀키 파일(PEM 파일) 필요\
PEM 파일 다운로드하기를 클릭하여 PEM 파일 저장

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/key2.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

### Command Prompt(혹은 Powershell) 실행

1. 윈도우(<span style="font-family: 'Wingdings';">&#xFF;</span>)  + R 키를 누르기
1. cmd 입력 (또는 powershell 입력)
1. Enter키 눌러서 Command Prompt 실행

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/key3.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

4. "`cd [PEM파일 폴더 경로]`" 명령어를 사용하여 PEM 파일이 있는 위치로 이동\
예시: `cd %userprofile%\Downloads`\
(PEM 저장 위치 예시: 다운로드 폴더, 바탕화면, USB 등...)

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/key4.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

### SSH 접속 명령어

1. 내 인스턴스 페이지에서 다른 SSH 클라이언트 사용 선택

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/ssh.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

2. 빠른 연결칸의 명령어를 복사

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/ssh2.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

3. 복사한 명령어를 앞서 실행한 powershell 또는 cmd에 붙여넣기

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/ssh3.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

4. SSH 접속 완료, SSH 접속 창은 계속 사용 되므로 창을 닫지 않음

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/ssh4.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

## 버전 확인

### 논문에서 구성한 버전 정보

- pytorch: 2.1.0 이상
- pytorch_lightning : 1.8.2
- torchvision : 0.16.0 이상
- tensorboard : 2.15.0 이상
- numpy : 1.23.5
- hydra-core : 1.3

### 파이썬 버전 확인

주피터 노트북에서 아래 명령어를 입력하여 확인하며 `엘리스 클라우드`의 주피터 노트북에 접속하여 입력함
> Tip: 파이썬 버전에 따라 라이브러리 설치 가능 버전이 달라짐

1. 내 인스턴스에서 연결을 클릭하여 Jupyter notebook에 접속

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/version.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

2. 우측 상단 `new` 버튼 클릭후 `Python 3(ipykernel)` 버튼 클릭

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/version2.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

3. 코드를 작성하여 파이썬 버전을 확인

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/version3.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

In [ ]:
import sys
print('Python version', sys.version)

## 코드 업로드

### 주피터 노트북 웹 페이지에서 바로 업로드 하기

1. 내 인스턴스에서 연결을 클릭하여 Jupyter notebook에 접속

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/version.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

2. 우측 상단에 있는 `Upload` 버튼 클릭

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/upload1.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

3. 열기 대화창이 뜨면 다운 받은 주피터노트북 교육자료 파일(*.ipynb) 선택 후 `열기(O)` 클릭

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/upload2.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

4. 파란색 `Upload` 버튼을 클릭하여 엘리스 클라우드 서버로 파일 업로드

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/upload3.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

5. 업로드한 교육자료 파일을 클릭하여 파일 열기

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/upload4.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

6. 강사의 지도에 따라 교육 자료 활용

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/upload5.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

## 패키지 설치

### Miniconda 설치

> 원활하게 다양한 실습환경을 구축하기 위해 Miniconda의 가상환경을 생성하여 실습을 진행

SSH 접속 후 아래의 명령어를 순차적으로 실행
- wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
- bash Miniconda3-latest-Linux-x86_64.sh
- source ~/.bashrc

1. 명령어 작성 예시

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/miniconda.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

2. Please, press Enter to continue가 나오면 Enter 키를 누른 후 Page Down 키를 눌러 가장 아래로 내려감

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/miniconda2.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

3. Do you accept the license terms? [yes/no]가 나오며 yes 입력 후 Enter 키를 눌러 넘어감

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/miniconda2-1.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

4. You can undo this by running 'conda init --reverse $SHELL'? [yes/no]가 나오면 yes를 입력하여 Miniconda 설치 완료

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/miniconda2-2.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

5. Miniconda 설치 완료\
source ~/.bashrc 를 입력하여 변경 사항을 적용

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/miniconda3.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

### Conda 가상환경 생성 및 활성화

1. SSH에 가상환경 생성 명령어 작성
- conda create -n sensorscan python=3.10
> conda create -n `가상환경 이름` python=`파이썬 버전`

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/CondaEnv.png?raw=true" width="900">
    <figcaption>가상환경 생성 명령어 작성 예시</figcaption>
</figure>

2. 가상 환경 실행
- conda activate sensorscan
> conda activate `가상환경 이름`

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/CondaEnv2.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

### Jupyter Notebook과 가상환경 연동

1. ipykernel 설치
- pip install ipykernel

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/CondaEnv3.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

2. Jupyter Notebook에서 이 가상환경을 선택 가능한 커널로 등록
- python -m ipykernel install --user --name=sensorscan --display-name "SensorSCAN"
> python -m ipykernel install --user --name=`가상환경 이름` --display-name "`Jupyter에서 보여지는 이름`"

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/CondaEnv4.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

3. Jupyter Notebook에서 python 3.8 버전으로 notebook 생성 가능

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/CondaEnv5.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

4. 기존에 생성된 notebook에서 커널 변경 가능

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/CondaEnv6.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

5. python 버전이 정상적으로 변경 되었는지 확인

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/CondaEnv7.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

In [ ]:
import sys
print('Python version', sys.version)

### CUDA 설정

1. 패키지 관리자 업데이트
- sudo apt-get update
<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/cuda1.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

2. CUDA Toolkit 설치
- sudo apt-get install -y cuda-toolkit-12-4

<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/cuda2.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

3. cuDNN 설치
- sudo apt-get
 install -y libcudnn9-cuda-12
<figure>
    <img src="https://github.com/KDT-ai/Book-Image/blob/main/image/cloud/cuda3.png?raw=true" width="900">
    <!-- <figcaption></figcaption> -->
</figure>

### GPU 확인

NVIDIA 드라이버 확인

In [ ]:
!nvidia-smi

CUDA 버전 확인

In [ ]:
!nvcc --version
# 12.4이어야 함
# 버전 넘버링은 넷째 줄 ... release [버전], ... 에서 확인할 수 있음

cuDNN 버전 확인

In [ ]:
!cat /usr/include/cudnn_version.h | grep CUDNN_MAJOR -A 2
# 8.9.6이어야 함
# 버전 넘버링은 [MAJOR.MINOR.PATCHLEVEL]임

Pytorch 설치

In [ ]:
!pip install torch

CUDA 지원 확인
- 정상적인 경우 출력 결과가 True로 표시됨
- 비정상적인 경우 출력 결과가 False로 표시됨

In [ ]:
import torch
print(torch.cuda.is_available())
# GPU 사용 가능 -> True, GPU 사용 불가 -> False

### 필요한 라이브러리

`요구 사항`
- joblib
- numpy : 1.23.5
- torch : 2.1.0 이상
- pytorch_lightning : 1.8.2
- torchvision : 0.16.0 이상
- tensorboard : 2.15.0 이상
- hydra-core : 1.3

### 패키지 버전 및 설치 확인

`필요 라이브러리 설치 명령어`

In [ ]:
!git clone https://github.com/AIRI-Institute/sensorscan.git

In [ ]:
cd /home/elicer/sensorscan

In [ ]:
!pip install -r requirements.txt

또는

In [ ]:
!git clone https://github.com/AIRI-Institute/sensorscan.git

In [ ]:
cd /home/elicer/sensorscan

In [ ]:
!pip install git+https://github.com/airi-industrial-ai/fddbenchmark@v0.0.3 aiohttp==3.11.16 certifi==2025.1.31 charset-normalizer==3.4.1 hydra-core==1.3.0 lightning-utilities==0.3.0 pandas==2.2.3 pytorch-lightning==1.8.2 pytz==2025.2 requests==2.32.3 scikit-learn==1.6.1 scipy==1.15.2 threadpoolctl==3.6.0 torch==2.6.0 torchmetrics==0.11.4 torchvision==0.21.0 tzdata==2025.2 urllib3==2.3.0

`주피터 노트북 환경에서 패키지 설치 여부 및 버전 확인`

In [ ]:
import torch
print(torch.__version__)

`Linux 환경에서 패키지 설치 여부 및 버전 확인`
- pip list | grep `패키지_이름`
- conda list | grep `패키지_이름`

> Tip: Windows는 grep 명령어 대신 findstr 명령어 사용\
> %pip list | findstr `패키지_이름`\
> %conda list | findstr `패키지_이름`

---
# 데이터셋 준비

## 데이터셋 소개

`Tennessee Eastman Process (TEP)`
- 실제 화학 공정을 모사(시뮬레이션)하여 생성된 데이터로, 공정 제어(process control) 및 결함 탐지/진단(Fault Detection and Diagnosis, FDD) 알고리즘을 테스트하고 비교하기 위한 표준 벤치마크임. 논문에서는 TEP의 두 가지 서로 다른 확장 데이터셋을 사용함. 정상 상태와 다양한 고장 상태에서의 공정 변화를 모두 담고 있으며, 데이터의 많은 부분이 '고장 상황'을 나타내는 기록함.

    - $TEP_{Rieth}$ / 파일명, 크기 : rieth_tep.zip, 1.98GB
        - TEP의 표준 시뮬레이션 설정(결함 목록 기반)을 사용하여 생성된 대규모의 확장된 벤치마크 데이터셋
    - $TEP_{Ricker}$ / 파일명, 크기 : reinartz_tep.zip, 2.01GB
        -  다른 제어 방식들과 수정된(revised) TEP 모델을 기반으로 생성된 확장 데이터셋

## 데이터셋 및 인공 신경망 API 다운로드

## 데이터셋 다운로드

아래 코드 순차적으로 수행
- 데이터셋 다운로드 경로 설정, 다운로드, 압축 해제 및 원래 경로로 되돌아가기

In [ ]:
!pip install gdown
# 구글드라이브 파일 다운로드 라이브러리

In [ ]:
mkdir data

In [ ]:
cd data

In [ ]:
!gdown "https://drive.google.com/uc?id=1bBN54_Vq0blcYc5godwqN39xn92gInYC" -O rieth_tep.zip && gdown "https://drive.google.com/uc?id=1bC8Ihc11_fmFkoLMKm7dhRnASY4dJl5S" -O reinartz_tep.zip
# 파일 다운로드

In [ ]:
!mkdir rieth_tep && mkdir reinartz_tep && unzip rieth_tep.zip -d /home/elicer/sensorscan/data/rieth_tep && unzip reinartz_tep.zip -d /home/elicer/sensorscan/data/reinartz_tep
# 경로 생성 및 압축 해제

In [ ]:
cd ..

## 데이터셋 확인

In [ ]:
import os
data_dir = "/home/elicer/sensorscan/data"
print('Labels:',len(os.listdir(data_dir)), os.listdir(data_dir))

## 데이터셋 시각화

`필요한 라이브러리`

In [ ]:
!pip install matplotlib
# 데이터 및 그래프 시각화 라이브러리

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

`데이터셋 시각화 함수 정의 및 시각화`

$TEP_{Rieth}$ 시각화 함수 정의

In [ ]:
def visualization_rieth_tep(path):
    dataset_name = "rieth_tep"
    columns_to_read = ['xmeas_1', 'xmeas_2', 'xmeas_3', 'xmv_1', 'xmv_2', 'xmv_3']

    data = pd.read_csv(f"{path}/{dataset_name}/dataset.csv", usecols=columns_to_read)

    fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    data[['xmeas_1', 'xmeas_2', 'xmeas_3']].plot(ax=axs[0], title=f"{dataset_name} - Measured Variables")
    axs[0].set_ylabel("xmeas values")

    data[['xmv_1', 'xmv_2', 'xmv_3']].plot(ax=axs[1], title=f"{dataset_name} - Manipulated Variables")
    axs[1].set_xlabel("Time")
    axs[1].set_ylabel("xmv values")

    plt.tight_layout()
    plt.show()

$TEP_{Ricker}$ 시각화 함수 정의

In [ ]:
def visualization_reinartz_tep(path):
    dataset_name = "reinartz_tep"
    columns_to_read = ['xmeas_1', 'xmeas_2', 'xmeas_3', 'xmv_1', 'xmv_2', 'xmv_3']

    data = pd.read_csv(f"{path}/{dataset_name}/dataset.csv", usecols=columns_to_read)

    fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    data[['xmeas_1', 'xmeas_2', 'xmeas_3']].plot(ax=axs[0], title=f"{dataset_name} - Measured Variables")
    axs[0].set_ylabel("xmeas values")

    data[['xmv_1', 'xmv_2', 'xmv_3']].plot(ax=axs[1], title=f"{dataset_name} - Manipulated Variables")
    axs[1].set_xlabel("Time")
    axs[1].set_ylabel("xmv values")

    plt.tight_layout()
    plt.show()

$TEP_{Rieth}$ 시각화

In [ ]:
visualization_rieth_tep("/home/elicer/sensorscan/data")

$TEP_{Ricker}$ 시각화

In [ ]:
visualization_reinartz_tep("/home/elicer/sensorscan/data")

# 모델 구축

## 파라미터 설정 및 데이터셋 설정

In [ ]:
!pip install easydict

In [ ]:
import yaml
import numpy as np
import random
import os
from easydict import EasyDict
import logging

target_dataset = "rieth_tep"
# target_dataset = "reinartz_tep"
config_base_path = "/home/elicer/sensorscan/configs/"
config_filename = f"sensorscan_{target_dataset}.yaml"
config_file_path = os.path.join(config_base_path, config_filename)
print(f"Attempting to load configuration from: {config_file_path}")

# YAML 파일 로드 및 파싱
with open(config_file_path, 'r') as f:
    config_dict = yaml.safe_load(f)
logging.info(f"Successfully loaded configuration from: {config_file_path}")
if 'pretraining' in config_dict and 'dataset_name' in config_dict['pretraining']:
    if config_dict['pretraining']['dataset_name'] == '${dataset}':
        config_dict['pretraining']['dataset_name'] = config_dict.get('dataset', target_dataset)

cfg = EasyDict(config_dict) # 설정 객체 (cfg) 생성

# 데이터셋에 따라 클러스터 수 동적 설정
if cfg.dataset == 'rieth_tep':
    cfg.num_clusters = 21
elif cfg.dataset == 'reinartz_tep':
    cfg.num_clusters = 29

# 재현성을 위한 시드 설정
random_seed = cfg.random_seed
if random_seed is not None:
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)
    random.seed(random_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_seed)

# 결정론적 연산 설정
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

# 장치 설정
cfg.device = 'cuda'
if 'pretraining' in cfg:
    cfg.pretraining.device = 'cuda'
    
print("Finish.")

## 필요한 모듈 로딩

In [ ]:
from tqdm.auto import tqdm
from tqdm.auto import tqdm
from typing import Any, Optional, Dict, List
from sklearn.neighbors import NearestNeighbors
from fddbenchmark import FDDDataset, FDDDataloader, FDDEvaluator
from models.sensorscan.model import build_encoder, build_clustering, SensorSCAN, init_weights
from models.sensorscan.data_utils import build_pretraining_dataloader, build_neighbour_loader
from models.sensorscan.optim import build_pretraining_optim, build_scan_optim
from models.sensorscan.train_utils import train_ssl_epoch, train_scan_epoch
import models.sensorscan
import math
import pandas as pd
import torch.nn.functional as F
import utils
import pandas as pd
import warnings

print("Finish.")

## 데이터셋 분할 및 로드

In [ ]:
logging.info('Creating dataset and applying preprocessing...')

# FDDDataset 객체 생성
dataset_name = cfg['dataset']
dataset = FDDDataset(name=dataset_name)
logging.info(f"Loaded dataset: {dataset_name}")
logging.info(f"Initial dataframe shape: {dataset.df.shape}")

original_columns = dataset.df.columns.tolist()
dataset.df = utils.exclude_columns(dataset.df)
logging.info(f"Dataframe shape after excluding columns: {dataset.df.shape}")
excluded_cols = set(original_columns) - set(dataset.df.columns.tolist())
logging.info(f"Excluded columns: {excluded_cols}")
utils.normalize(dataset) # 데이터 정규화

# Rieth TEP 데이터셋 불균형 조정
if dataset_name == "rieth_tep":
    logging.info("Applying Rieth TEP imbalance adjustment...")
    original_train_samples = dataset.train_mask.sum()
    dataset.train_mask = utils.make_rieth_imbalance(dataset.train_mask)
    new_train_samples = dataset.train_mask.sum()

# 학습 데이터 로더 생성
train_loader = FDDDataloader(
    dataframe=dataset.df,
    mask=dataset.train_mask,
    label=dataset.label,
    window_size=cfg['window_size'],
    step_size=cfg['step_size'],
    use_minibatches=True, # 미니배치 사용 여부
    batch_size=cfg['eval_batch_size'], # 평가 시 배치 크기 사용
    shuffle=True, # 학습 시 데이터 섞기
)

# 테스트 데이터 로더 생성
test_loader = FDDDataloader(
    dataframe=dataset.df,
    mask=dataset.test_mask,
    label=dataset.label,
    window_size=cfg['window_size'],
    step_size=cfg['step_size'],
    use_minibatches=True,
    batch_size=cfg['eval_batch_size'],
    shuffle=False, # 평가 시 데이터 섞지 않음
)

print("Finish.")

## 데이터셋 분할 결과 확인

In [ ]:
print(f"Dataset Name: {target_dataset}")
print(f"Full preprocessed DataFrame shape: {dataset.df.shape}")
print(f"Number of training samples (True values in mask): {dataset.train_mask.sum()}")
print(f"Number of testing samples (True values in mask): {dataset.test_mask.sum()}")
print(f"Label Series shape: {dataset.label.shape}")
print("Finish.")

## 모델 생성 함수

In [ ]:
# 인코더 생성 (사전 학습 설정값 사용)
encoder = build_encoder(cfg['pretraining'])

# 클러스터링 모델 생성
clustering_model = build_clustering(cfg)

print("Finish.")

## 모델 학습 함수

In [ ]:
warnings.filterwarnings("ignore", message="Creating an ndarray from ragged nested sequences")

cfg.pretraining.lr = float(cfg.pretraining.lr)
cfg.pretraining.weight_decay = float(cfg.pretraining.weight_decay)
cfg.encoder_lr = float(cfg.encoder_lr)
cfg.clustering_lr = float(cfg.clustering_lr)

# 모델 저장 경로 확인 및 생성
model_save_dir = 'saved_models'
os.makedirs(model_save_dir, exist_ok=True)

current_dataset_name = cfg.dataset
default_model_filename = f'sensorscan_{current_dataset_name}.pth'
cfg.path_to_model = os.path.join(model_save_dir, default_model_filename)
logging.info(f"Model will be trained and saved to: {cfg.path_to_model}")

# 1. 사전 학습
logging.info('Starting Pretraining phase...')
pretraining_loader = build_pretraining_dataloader(cfg.pretraining)
logging.info(f"Pretraining dataloader built. Batches: {len(pretraining_loader)}")
loss_fn_ssl, optimizer_ssl = build_pretraining_optim(cfg.pretraining, encoder)
logging.info("Pretraining optimizer and loss function built.")
for epoch in range(cfg.pretraining.epochs):
    avg_loss = train_ssl_epoch(cfg.pretraining, encoder, pretraining_loader, loss_fn_ssl, optimizer_ssl)
    logging.info(f'Pretraining Epoch {epoch+1}/{cfg.pretraining.epochs}: Average Loss = {avg_loss:.6f}')
logging.info('Pretraining finished.')

# 2. SCAN 학습
logging.info('Starting SCAN training phase...')
neighbor_loader = build_neighbour_loader(cfg, encoder)
logging.info(f"Neighbor dataloader built. Batches: {len(neighbor_loader)}")
loss_fn_scan, encoder_optimizer_scan, clustering_optimizer_scan = build_scan_optim(cfg, encoder, clustering_model)
logging.info("SCAN optimizer and loss function built.")
for epoch in range(cfg.epochs):
    avg_loss = train_scan_epoch(cfg, epoch, encoder, clustering_model, neighbor_loader, loss_fn_scan, encoder_optimizer_scan, clustering_optimizer_scan)
    logging.info(f'SCAN Training Epoch {epoch+1}/{cfg.epochs}: Average Loss = {avg_loss:.6f}')
logging.info('SCAN training finished.')

# 3. 모델 결합 및 저장
logging.info("Combining trained components into SensorSCAN model...")
sensorscan_trained = SensorSCAN(encoder, clustering_model, device=cfg.device)
logging.info(f"Saving trained model to {cfg.path_to_model}...")
torch.save(sensorscan_trained.state_dict(), cfg.path_to_model)
logging.info("Model saved successfully.")

# 4. 최종 모델 로드
logging.info("Loading the final SensorSCAN model (just trained) for evaluation...")
sensorscan = SensorSCAN(build_encoder(cfg.pretraining), build_clustering(cfg), device=cfg.device)
sensorscan.load_state_dict(torch.load(cfg.path_to_model, map_location=cfg.device))

sensorscan.eval()
logging.info("SensorSCAN model is ready for evaluation.")

print("Finish.")

## 모델 평가 함수

In [ ]:
logging.info('Getting predictions on train set...')
sensorscan.eval()
train_pred_list = []
train_label_list = []

# 학습 데이터 예측
with torch.no_grad():
    for X, time_index, label in tqdm(train_loader, desc='Predicting on train set'):
        X = torch.FloatTensor(X).to(cfg['device'])
        pred = sensorscan(X).cpu().numpy().argmax(1)
        train_pred_list.append(pd.Series(pred, index=time_index))
        train_label_list.append(pd.Series(label, index=time_index))

# 예측 결과와 레이블을 하나의 Series로 합침
train_pred = pd.concat(train_pred_list)
train_label = pd.concat(train_label_list).astype('int')
logging.info(f"Train predictions generated. Length: {len(train_pred)}")
logging.info('Getting predictions on test set...')
test_pred_list = []
test_label_list = []

# 테스트 데이터 예측
with torch.no_grad():
    for X, time_index, label in tqdm(test_loader, desc='Predicting on test set'):
        X = torch.FloatTensor(X).to(cfg['device'])
        pred = sensorscan(X).cpu().numpy().argmax(1)
        test_pred_list.append(pd.Series(pred, index=time_index))
        test_label_list.append(pd.Series(label, index=time_index))

# 예측 결과와 레이블을 하나의 Series로 합침
test_pred = pd.concat(test_pred_list)
test_label = pd.concat(test_label_list).astype('int')
logging.info(f"Test predictions generated. Length: {len(test_pred)}")

# 성능 지표 계산 및 출력
logging.info('Calculating evaluation metrics...')

# FDDEvaluator 초기화
evaluator = FDDEvaluator(step_size=cfg['step_size'])

# 1. 클러스터링 성능 평가
logging.info('Calculating clustering metrics (using test set predictions before label matching)')
clustering_metrics = evaluator.evaluate(test_label, test_pred)
utils.print_clustering(clustering_metrics, logging)

# 2. 레이블 매칭
logging.info('Creating label matching using weighted max occurence (train set)...')
label_matching = utils.weighted_max_occurence(train_label, train_pred, n_types)
logging.info(f"Label matching created: {label_matching}")

# 3. 매칭된 레이블로 테스트 예측 결과 변환
logging.info('Remapping test predictions using the label matching...')
remapped_test_pred = pd.Series(label_matching[test_pred.values], index=test_pred.index)

# 4. 고장 진단 성능 평가
logging.info('Calculating FDD metrics (using remapped test set predictions)')
fdd_metrics = evaluator.evaluate(test_label, remapped_test_pred)
utils.print_fdd(fdd_metrics, logging)

print("Finish.")

# 모델 학습

## 주요 변수 설명

- cfg : 모델 학습 및 설정에 필요한 모든 하이퍼파라미터와 경로 등을 담은 객체.
- encoder : 시계열 데이터의 특징을 추출하는 인코더 모델 부분.
- clustering_model : 인코더 특징을 받아 클러스터링(고장 분류)을 수행하는 모델 부분.
- pretraining_loader : 사전 훈련 단계에서 사용할 데이터 배치(batch)를 제공하는 로더.
- loss_fn_ssl : 사전 훈련 단계의 손실 함수 (재구성 손실 + 대조 손실).
- neighbor_loader : SCAN 훈련 단계에서 샘플과 이웃 샘플 쌍을 제공하는 데이터 로더.
- loss_fn_scan : SCAN 훈련 단계의 손실 함수 (일관성 손실 + 엔트로피 손실).
- Sensorscan : 최종적으로 평가(Evaluation)에 사용될 SensorSCAN 모델 인스턴스.


# 모델 실험 결과

## 결과 분석

- 두 개의 다른 데이터셋에 대해 네 가지 방법의 성능을 비교한 실험
- 각 방법에 대해 ACC, ARI, NMI, Detection TPR 등 7가지 평가지표를 사용하여 성능을 측정함
- 기존의 성능이 제일 좋은 모델(convae)보다 성능이 좋고(ACC, ARI 등) 추론 속도도 빠른(ADD) 모습을 보임

# 결론

## 주요 업적

### SensorSCAN
- 완전 비지도 FDD 성능 확보
    - 레이블 없는 상태에서 Detection TPR 0.87, FPR≈0, CDR 0.96, ADD 28.47 스텝을 기록하며, PCA·ConvAE·ST-CatGAN 대비 TPR을 0.20–0.30p.p. 이상 개선

- 고밀도 임베딩 기반 클러스터링
    - Transformer-기반 self-supervised 사전학습과 변형된 SCAN 클러스터링을 결합해, 클러스터링 지표(ACC=0.785, ARI=0.703, NMI=0.846)에서 현존 기법을 크게 앞섬

- 극소량 라벨로 지도급 성능 달성
    - “한 번의 라벨링 런” 수준만으로 미세조정해, 전체 라벨을 사용하는 GRU 풀 학습 모델과 동등한 탐지 정확도를 구현

- 동적 클러스터 수 설정 지원
    - 데이터셋별로 최적의 클러스터 수를 자동 반영(rieth=21개, reinartz=29개)하여, 멀티프로세스·멀티환경에 유연하게 적용

### 실무 적용
- 실시간 모니터링 구현
    - 샘플당 예측 시간 ≈0.01초로, TEP 공정 3분 주기 대비 여유롭게 실시간 경보 발송 가능 .

- 엣지 디바이스 배포 용이
    - 모델 파라미터 크기 2.38 MB, 낮은 메모리·연산 요구량으로 현장 하드웨어 부담 최소화 

- 최소한의 전문가 개입
    - 가중 최대 빈도 기반 자동 Label Matching 기법으로 클러스터→상태 매핑을 자동화해 초기 설정 부담을 경감 

- 빠른 PoC 및 확장성
    - 오픈소스 구현체를 Docker/컨테이너로 즉시 배포 가능하며, 신규 결함 유형 추가 시에도 파이프라인 재실행과 소량 라벨만으로 빠르게 대응

## 현장 실무에 유용할 주요 핵심 정리

- 무라벨→배포 단일 워크플로우
    - 레이블 없이 학습→클러스터링→미세조정→배포까지 일관된 파이프라인

- 소량 라벨로 지도급 정확도
    - 극소량 라벨로도 SOTA급 FDD 달성

- 초저지연·저자원 예측
    - 0.01초, <3 MB로 현장 HW 제약 완화

- 자동화된 클러스터-레이블 매핑
    - 전문가 개입 최소화

- 실시간 대시보드 연동
    - SCADA/ERP 시스템과의 손쉬운 통합으로 즉시 경보·보고 가능


## 기존 연구와의 차별점

- 완전 비지도 FDD 파이프라인
    - SensorSCAN은 레이블이 전혀 없는 상태에서 self-supervised pretraining과 변형된 SCAN 딥 클러스터링을 결합해, 센서 시계열 데이터의 이상 탐지·진단을 수행
    - 기존의 PCA·ConvAE·ST-CatGAN 등은 재구성 또는 통계 기반 분리 기법에 그쳤으나, 본 연구는 강화된 표현력의 Transformer 임베딩과 클러스터링 헤드를 end-to-end로 통합

- 자동화된 클러스터-레이블 매핑
    - 전통적 방식은 클러스터 결과를 전문가가 수작업으로 매핑해야 했으나, 가중 최대 빈도(weighted max-occurrence) 기반 Label Matching 기법을 도입해 최소한의 검토만으로 클러스터에 정상·결함 상태를 자동 할당

- 소량 라벨로 지도학습급 성능
    - “한 번의 라벨링 런” 수준(극소량)만으로 미세조정(fine-tuning)해, 전체 라벨을 요구하는 GRU 기반 SOTA 모델에 근접한 결함 탐지·진단 정확도를 달성


## 기술적 한계 및 향후 연구 과제

### 기술적 한계
- 일부 결함에 대한 탐지 한계
    - 완전 비지도 단계에서 특정 결함(예: Fault 15, 21)은 TPR이 0%에 가까워 지속적으로 놓치는 사례가 있음

- 클러스터 매핑 의존성
    - Label Matching은 클러스터 내 샘플 분포 비율에 기반하므로, 비정형 분포나 소수 샘플 클러스터의 경우 잘못 매핑될 위험이 있음

- 연산·메모리 부담
    - SSL 사전학습과 대규모 클러스터링 단계는 컴퓨팅 자원 및 학습 시간이 비교적 많이 필요해, 엣지 환경 적용 시 제약이 있을 수 있음

- 클러스터 수 사전 결정 필요
    - 현재 구현은 데이터셋별 클러스터 수(예: TEPRieth 21개, TEPRicker 29개)를 사전에 지정해야 하며, 최적값 미설정 시 성능이 저하될 수 있음

- 범용성 검증 한정적
    - TEP 공정 벤치마크에만 평가되었고, 다른 산업 공정이나 센서 어레이에는 추가 실험이 필요


### 향후 연구 과제
- 도메인 지식 통합 SSL 과제
    - 화학·물리 법칙을 활용한 마스킹 복원 및 대조 학습 과제를 설계해, 공정 특유의 구조적 제약을 임베딩에 반영할 필요가 있음

- 심층 반지도 학습 기법 도입
    - 최신 semi-supervised 시계열 분류 방법([96–98])을 FDD 미세조정에 적용해, 어려운 결함까지 효과적으로 탐지하는 연구가 요구됨

- 적응적 클러스터링 메커니즘
    - 동적 클러스터 수 결정 또는 비모수적 군집화(예: Dirichlet 프로세스 기반)로 클러스터 수 불확실성을 해소하는 방안 검토

- 시공간 의존성 강화 모델
    - 그래프 신경망(GNN)·TCN 등 시계열의 시간적·공간적 상관관계를 명시적으로 반영하는 구조 연구

- 다중 프로세스·다중 뷰 적용
    - 다양한 공정 및 센서 모달리티로 범용성 검증을 수행하고, 멀티태스크 학습으로 여러 공정을 동시 진단하는 확장 가능성 탐색